In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

In [30]:
#Load Dataset
df = pd.read_csv("../data/PS_20174392719_1491204439457_log.csv")

In [31]:
#Quick check
print("Dataset shape:",df.shape)
print(df.head())
print(df['type'].value_counts())


Dataset shape: (6362620, 11)
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER   

In [32]:
# Filter only Cash_out and Transfer transactions
df = df[df['type'].isin(['CASH_OUT', 'TRANSFER'])]
print("Filtered Shape:", df.shape)


Filtered Shape: (2770409, 11)


In [33]:
# Encode transaction type to numbers
le = LabelEncoder()
df['type_encoded'] = le.fit_transform(df['type'])
# Check encoding
print(df['type_encoded'].unique())

[1 0]


In [34]:
# 4️⃣ Derive Features
# --- Balance Drain Features ---
df['drain_score'] = (df['oldbalanceOrg'] - df['newbalanceOrig']) / (df['oldbalanceOrg'] + 1)
df['dest_balance_discrepancy'] = abs((df['oldbalanceDest'] + df['amount']) - df['newbalanceDest'])
df['origin_zeroed'] = (df['newbalanceOrig'] == 0).astype(int)
df['dest_was_empty'] = (df['oldbalanceDest'] == 0).astype(int)

# --- Amount Behavior ---
df['amount_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)
df['is_large_amount'] = (df['amount'] > df['amount'].quantile(0.95)).astype(int)
df['is_round_amount'] = (df['amount'] % 1000 == 0).astype(int)

# --- Time Features ---
df['hour_of_day'] = df['step'] % 24
df['day_of_week'] = (df['step'] // 24) % 7
df['is_night'] = df['hour_of_day'].between(0, 6).astype(int)
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)



In [35]:
# ==========================================
# 5️⃣ 🌟 5 UNIQUE FEATURES
# ==========================================

# 🌟 #1 — Phantom Money Score
# Money that left sender but never arrived at receiver
# Score near 1 = money vanished = strong fraud signal
df['phantom_money_score'] = (
    (df['oldbalanceOrg'] - df['newbalanceOrig']) -
    (df['newbalanceDest'] - df['oldbalanceDest'])
) / (df['amount'] + 1)

# 🌟 #2 — Balance Wipeout Ratio
# What % of sender's total worth was moved in this one transaction
# Score near 1.0 = almost total account wipeout
df['balance_wipeout_ratio'] = df['amount'] / (df['oldbalanceOrg'] + df['amount'] + 1)

# 🌟 #3 — Zero-to-Zero Flag
# Sender ends at zero AND receiver ends at zero
# Classic money laundering / mule chain pattern
df['zero_to_zero_flag'] = (
    (df['newbalanceOrig'] == 0) & (df['newbalanceDest'] == 0)
).astype(int)

# 🌟 #4 — Exact Drain Flag
# Fraudster takes EXACTLY what's in the account to the penny
# Normal humans don't do this — automated scripts do
df['exact_drain_flag'] = (
    abs(df['amount'] - df['oldbalanceOrg']) < 1.0
).astype(int)

# 🌟 #5 — Mule Account Score
# Receiver was empty before receiving a large transfer
# Score near 1 = disposable mule account pattern
df['mule_account_score'] = df['amount'] / (df['oldbalanceDest'] + df['amount'] + 1)



In [36]:
# 5PLIT BY SENDERR
senders = df['nameOrig'].unique()
train_senders, test_senders = train_test_split(senders,test_size=0.2, random_state=42)

train_df = df[df['nameOrig'].isin(train_senders)].copy()
test_df = df[df['nameOrig'].isin(test_senders)].copy()
print(f"\nTrain size: {len(train_df):,} | Test size: {len(test_df):,}")


Train size: 2,216,303 | Test size: 554,106


In [37]:
# ==========================================
# 7️⃣ AGGREGATION FEATURES (Train only → merge)
# ==========================================

# Sender stats
sender_stats = train_df.groupby('nameOrig')['amount'].agg(
    tx_count_by_sender='count',
    total_sent_by_sender='sum',
    sender_avg_amount='mean',
    sender_std_amount='std'
).reset_index()

# Receiver stats
receiver_stats = train_df.groupby('nameDest')['amount'].agg(
    recv_tx_count='count',
    recv_total_received='sum'
).reset_index()

# Time velocity
train_df = train_df.sort_values(['nameOrig', 'step'])
train_df['time_since_last_tx'] = train_df.groupby('nameOrig')['step'].diff().fillna(999)
train_df['rapid_fire_flag']    = (train_df['time_since_last_tx'] < 1).astype(int)

test_df = test_df.sort_values(['nameOrig', 'step'])
test_df['time_since_last_tx']  = test_df.groupby('nameOrig')['step'].diff().fillna(999)
test_df['rapid_fire_flag']     = (test_df['time_since_last_tx'] < 1).astype(int)

# Merge into both sets (train stats only)
train_df = train_df.merge(sender_stats,   on='nameOrig', how='left')
train_df = train_df.merge(receiver_stats, on='nameDest',  how='left')

test_df  = test_df.merge(sender_stats,   on='nameOrig', how='left')
test_df  = test_df.merge(receiver_stats, on='nameDest',  how='left')

# Amount deviation (needs sender_avg_amount to exist first)
train_df['amount_deviation'] = abs(train_df['amount'] - train_df['sender_avg_amount']) / (train_df['sender_std_amount'] + 1)
test_df['amount_deviation']  = abs(test_df['amount']  - test_df['sender_avg_amount'])  / (test_df['sender_std_amount']  + 1)

# Fill NaN for unseen senders/receivers in test
train_df.fillna(0, inplace=True)
test_df.fillna(0, inplace=True)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,...,mule_account_score,time_since_last_tx,rapid_fire_flag,tx_count_by_sender,total_sent_by_sender,sender_avg_amount,sender_std_amount,recv_tx_count,recv_total_received,amount_deviation
0,307,TRANSFER,673726.47,C1000008236,307.0,0.0,C107425672,0.00,673726.47,0,...,0.999999,999.0,0,0.0,0.0,0.0,0.0,4.0,881451.37,0.0
1,280,CASH_OUT,58347.84,C1000008393,10794.0,0.0,C615558732,2551590.32,2609938.16,0,...,0.022356,999.0,0,0.0,0.0,0.0,0.0,9.0,3016885.72,0.0
2,20,CASH_OUT,315626.96,C1000008582,0.0,0.0,C1462566442,1005922.02,1321548.98,0,...,0.238831,999.0,0,0.0,0.0,0.0,0.0,10.0,2774057.26,0.0
3,154,TRANSFER,235163.72,C1000009606,0.0,0.0,C1311531655,779432.65,1014596.37,0,...,0.231780,999.0,0,0.0,0.0,0.0,0.0,8.0,2115197.82,0.0
4,328,TRANSFER,165944.75,C1000010387,0.0,0.0,C740868212,758287.33,924232.08,0,...,0.179549,999.0,0,0.0,0.0,0.0,0.0,3.0,676900.79,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554101,283,CASH_OUT,288938.46,C999973040,0.0,0.0,C480277152,6619941.95,6908880.42,0,...,0.041821,999.0,0,0.0,0.0,0.0,0.0,22.0,13629598.23,0.0
554102,16,CASH_OUT,359526.22,C99997916,140556.0,0.0,C32023431,243884.21,603410.43,0,...,0.595823,999.0,0,0.0,0.0,0.0,0.0,7.0,840347.72,0.0
554103,378,CASH_OUT,26015.81,C999987705,0.0,0.0,C476434444,227337.98,253353.80,0,...,0.102685,999.0,0,0.0,0.0,0.0,0.0,3.0,286822.90,0.0
554104,227,TRANSFER,48713.45,C999989413,0.0,0.0,C1675562483,1666639.53,1715352.98,0,...,0.028398,999.0,0,0.0,0.0,0.0,0.0,11.0,6193682.55,0.0


In [38]:
# 7️⃣ Select features and target
features = [
    # Core
    'type_encoded', 'amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',

    # Standard derived
    'drain_score', 'dest_balance_discrepancy',
    'origin_zeroed', 'dest_was_empty',
    'amount_ratio', 'is_large_amount', 'is_round_amount',
    'hour_of_day', 'is_night',

    # 🌟 5 Unique features
    'phantom_money_score',
    'balance_wipeout_ratio',
    'zero_to_zero_flag',
    'exact_drain_flag',
    'mule_account_score',

    # Aggregation features
    'tx_count_by_sender', 'total_sent_by_sender',
    'sender_avg_amount', 'sender_std_amount', 'amount_deviation',
    'recv_tx_count', 'recv_total_received',
    'time_since_last_tx', 'rapid_fire_flag',
]
target = 'isFraud'


In [39]:
X_train = train_df[features]
y_train = train_df['isFraud']

X_test  = test_df[features]
y_test  = test_df['isFraud']

print(f"\nTotal features: {len(features)}")
print(f"Fraud rate in train: {y_train.mean()*100:.2f}%")
print(f"Fraud rate in test:  {y_test.mean()*100:.2f}%")


Total features: 29
Fraud rate in train: 0.29%
Fraud rate in test:  0.31%


In [40]:
# Train Random Forest
print("\nTraining model...")
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)


Training model...


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [41]:
#  PREDICTIONS
y_pred  = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]


In [42]:
# ==========================================
# 1️⃣1️⃣ EVALUATION
# ==========================================

print("\n" + "="*50)
print("   RANDOM FOREST MODEL vs ACTUAL FRAUD")
print("="*50)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")


# ==========================================
# 1️⃣2️⃣ 3-LEVEL RISK CLASSIFICATION
# ==========================================

def classify_transaction(prob):
    if prob >= 0.55:
        return "FRAUD ALERT 🚨"
    elif prob >= 0.15:
        return "SUSPICIOUS ⚠️"
    else:
        return "LEGIT ✅"

predicted_labels = [classify_transaction(p) for p in y_proba]


# ==========================================
# 1️⃣3️⃣ COMPARE WITH BANK RULE (isFlaggedFraud)
# ==========================================

print("\n" + "="*50)
print("   isFlaggedFraud vs ACTUAL FRAUD")
print("="*50)
print("Confusion Matrix:\n", confusion_matrix(y_test, test_df['isFlaggedFraud']))
print("\nClassification Report:\n", classification_report(y_test, test_df['isFlaggedFraud']))



   RANDOM FOREST MODEL vs ACTUAL FRAUD
Confusion Matrix:
 [[552390      1]
 [     6   1709]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    552391
           1       1.00      1.00      1.00      1715

    accuracy                           1.00    554106
   macro avg       1.00      1.00      1.00    554106
weighted avg       1.00      1.00      1.00    554106

ROC-AUC Score: 0.9993

   isFlaggedFraud vs ACTUAL FRAUD
Confusion Matrix:
 [[552391      0]
 [  1710      5]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    552391
           1       1.00      0.00      0.01      1715

    accuracy                           1.00    554106
   macro avg       1.00      0.50      0.50    554106
weighted avg       1.00      1.00      1.00    554106



In [43]:
# ==========================================
# 1️⃣5️⃣ RESULTS SUMMARY
# ==========================================

results = pd.DataFrame({
    "isFraud":         y_test.values,
    "fraud_prob":      y_proba.round(3),
    "predicted_label": predicted_labels,
    "isFlaggedFraud":  test_df['isFlaggedFraud'].values
})

print("\nSample Predictions:")
print(results.head(10))

print("\n=== Label Distribution ===")
print(pd.Series(predicted_labels).value_counts())


Sample Predictions:
   isFraud  fraud_prob predicted_label  isFlaggedFraud
0        0       0.033         LEGIT ✅               0
1        0       0.010         LEGIT ✅               0
2        0       0.155   SUSPICIOUS ⚠️               0
3        0       0.151   SUSPICIOUS ⚠️               0
4        0       0.135         LEGIT ✅               0
5        0       0.046         LEGIT ✅               0
6        0       0.043         LEGIT ✅               0
7        0       0.142         LEGIT ✅               0
8        0       0.063         LEGIT ✅               0
9        0       0.132         LEGIT ✅               0

=== Label Distribution ===
LEGIT ✅          467978
SUSPICIOUS ⚠️     84418
FRAUD ALERT 🚨      1710
Name: count, dtype: int64


In [44]:
# 1️⃣1️⃣ Save model
import os
import joblib
os.makedirs("../models", exist_ok=True)
model_file = "../models/random_forest_model_realistic.pkl"
joblib.dump(rf_model, model_file)
print(f"Model saved at {model_file}")

Model saved at ../models/random_forest_model_realistic.pkl
